# Répartition en train / validation / test / eval_clinique

Objectif : répartir l'agrégat SFT nettoyé (`notebooks/02_nettoyage_dedoublonnage.ipynb`) en quatre jeux avec `assign_splits` (dans `scripts/extraction.py`), et vérifier que la répartition est cohérente avant de passer au sous-échantillonnage à 5000 paires.

La répartition est stratifiée par source (mediqal, frenchmedmcqa, medquad) : chaque source suit les mêmes proportions de split, pour éviter qu'un split se retrouve dominé par une seule source ou une seule langue. Le split déjà présent sur FrenchMedMCQA (hérité du dataset source) est écrasé pour que toutes les sources suivent la même logique. Les proportions retenues : 80% train, 10% validation, 5% test, 5% eval_clinique. Ce choix est documenté dans `docs/decisions.md`.

In [1]:
import sys
sys.path.append("..")

from collections import Counter
from dotenv import load_dotenv
from scripts.extraction import build_sft_dataset

load_dotenv()

/home/rapha/ia-engineer/llm-finetuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Répartition globale et par source

In [2]:
agregat = build_sft_dataset()
print("agrégat total :", len(agregat))

repartition = Counter(r["split"] for r in agregat)
for split, n in repartition.items():
    print(f"{split:15s} {n:6d}  ({n / len(agregat):.1%})")

agrégat total : 22407
train            17925  (80.0%)
validation        2241  (10.0%)
eval_clinique     1121  (5.0%)
test              1120  (5.0%)


In [3]:
repartition_par_source = Counter((r["source"], r["split"]) for r in agregat)
sources = sorted(set(r["source"] for r in agregat))
splits = ["train", "validation", "test", "eval_clinique"]

for source in sources:
    total_source = sum(n for (s, _), n in repartition_par_source.items() if s == source)
    print(source, "-", total_source, "exemples")
    for split in splits:
        n = repartition_par_source[(source, split)]
        print(f"  {split:15s} {n:6d}  ({n / total_source:.1%})")

frenchmedmcqa - 1079 exemples
  train              863  (80.0%)
  validation         108  (10.0%)
  test                54  (5.0%)
  eval_clinique       54  (5.0%)
mediqal - 4969 exemples
  train             3975  (80.0%)
  validation         497  (10.0%)
  test               248  (5.0%)
  eval_clinique      249  (5.0%)
medquad - 16359 exemples
  train            13087  (80.0%)
  validation        1636  (10.0%)
  test               818  (5.0%)
  eval_clinique      818  (5.0%)


Les proportions par source suivent bien les ratios visés (80/10/5/5), aux arrondis près sur les petites sources.

## Vérification : pas de fuite entre splits

Comme la répartition se fait par simple partition des indices après mélange, il ne peut pas y avoir de recouvrement d'IDs entre splits. On le vérifie quand même explicitement, pour garder une preuve dans ce notebook.

In [4]:
ids_par_split = {split: set(r["id"] for r in agregat if r["split"] == split) for split in splits}

for i, split_a in enumerate(splits):
    for split_b in splits[i + 1:]:
        intersection = ids_par_split[split_a] & ids_par_split[split_b]
        print(f"{split_a} / {split_b} : {len(intersection)} id en commun")

train / validation : 0 id en commun
train / test : 0 id en commun
train / eval_clinique : 0 id en commun
validation / test : 0 id en commun
validation / eval_clinique : 0 id en commun
test / eval_clinique : 0 id en commun


## Synthèse

L'agrégat SFT est réparti en train, validation, test et eval_clinique, stratifié par source, sans aucune fuite d'ID entre splits. Prochaine étape : sous-échantillonner à environ 5000 paires en respectant ces proportions, puis anonymiser avec Presidio.